# **Librerias**

In [47]:
import pandas as pd
import numpy as np
import Levenshtein
import unicodedata
from itertools import combinations

# **Carga dataset**

In [48]:
# Catálogo: 8747 libros con slug, title, authors, genres, synopsis y otros metadatos. 
# Algunos registros están contaminados; no se indica cuáles. 
catalogo = pd.read_json("Fuente de datos/catalogo_tp3.jsonl", lines=True)

# Muestra: 500 libros del mismo catálogo con la columna contaminado (1 o 0). 
# En los contaminados, la columna fragmento contiene el texto que se insertó. 
muestra = pd.read_csv("Fuente de datos/muestra_etiquetada.csv", keep_default_na=False)

In [49]:
# Renombramos columnas en español
reemplazo_columnas = {
    "source_url": "url",
    "slug": "slug",
    "title": "titulo",
    "authors": "autores",
    "genres": "generos",
    "seed_genre": "generos_semilla",
    "series": "series",
    "series_number": "numero_serie",
    "cover_url": "url_portada",
    "synopsis": "sinopsis",
    "synopsis_chars" : "cantidad_caracteres_sinopsis",
    "scraped_at": "fecha_scrapeo"}
catalogo.rename(columns=reemplazo_columnas, inplace=True)

# Añadimos una columna con la fecha de scrapeo en formato datetime para poder filtrar por fecha
catalogo["fecha_scrapeo"] = pd.to_datetime(catalogo["fecha_scrapeo"])

# Mostramos las primeras filas del catálogo
catalogo.head()

,url,slug,titulo,autores,generos,generos_semilla,series,numero_serie,url_portada,sinopsis,cantidad_caracteres_sinopsis,fecha_scrapeo
0,https://ww3.lectulandia.com/book/de-la-caverna...,de-la-caverna-al-cosmos,De la caverna al cosmos,[Eudald Carbonell],"[Arqueología, Divulgación, Ensayo]",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/Eudald%20C...,La Humanidad ha entrado en una nueva fase evol...,998,2026-10-17 23:11:56+00:00
1,https://ww3.lectulandia.com/book/quienes-somos...,quienes-somos-y-como-hemos-llegado-hasta-aqui,Quiénes somos y cómo hemos llegado hasta aquí,[David Reich],"[Arqueología, Biología, Ciencias naturales, En...",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/David%20Re...,La extracción de ADN de los huesos de nuestros...,1285,2026-10-17 23:11:58+00:00
2,https://ww3.lectulandia.com/book/teotihuacan/,teotihuacan,Teotihuacan,[Eduardo Matos Moctezuma],"[Arqueología, Arte, Ciencias sociales, Ensayo,...",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/Eduardo%20...,Teotihuacan es la primera gran ciudad del cent...,1070,2026-10-17 23:12:01+00:00
3,https://ww3.lectulandia.com/book/tiro-y-las-co...,tiro-y-las-colonias-fenicias-de-occidente,Tiro y las colonias fenicias de Occidente,[M.ª Eugenia Aubet],"[Arqueología, Economía, Ensayo, Historia, Polí...",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/M%20Eugeni...,Entre los siglos VIII y VI a. C. los fenicios ...,1875,2026-10-17 23:12:03+00:00
4,https://ww3.lectulandia.com/book/la-maldicion-...,la-maldicion-del-hombre-mono,La maldición del hombre mono,[Emiliano Bruner],"[Arqueología, Ensayo, Psicología, Sociología]",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/Emiliano%2...,Las raíces evolutivas del sufrimiento humano\n...,1613,2026-10-17 23:12:05+00:00


# **EDA** (Analisis Exploratorio de Datos)

In [50]:
# Eliminamos registros sin sinopsis y duplicados por identificador. 
catalogo.dropna(subset=["sinopsis"], inplace=True)

# Vemos ahora informacion sobre el catalogo
catalogo.info()

<class 'pandas.DataFrame'>
Index: 8746 entries, 0 to 8746
Data columns (total 12 columns):
 #   Column                        Non-Null Count  Dtype              
---  ------                        --------------  -----              
 0   url                           8746 non-null   str                
 1   slug                          8746 non-null   str                
 2   titulo                        8746 non-null   str                
 3   autores                       8746 non-null   object             
 4   generos                       8746 non-null   object             
 5   generos_semilla               8746 non-null   str                
 6   series                        2316 non-null   str                
 7   numero_serie                  2298 non-null   float64            
 8   url_portada                   8746 non-null   str                
 9   sinopsis                      8746 non-null   str                
 10  cantidad_caracteres_sinopsis  8746 non-null   int64 

In [51]:
# Dimensiones de nuestro df:
catalogo.shape

(8746, 12)

## **Limpieza del texto** -> normalización

In [52]:
# Identificamos registros con saltos de línea en la sinopsis
catalogo[catalogo["sinopsis"].str.contains("\n", regex=True, na=False)]["sinopsis"]

0       La Humanidad ha entrado en una nueva fase evol...
1       La extracción de ADN de los huesos de nuestros...
2       Teotihuacan es la primera gran ciudad del cent...
3       Entre los siglos VIII y VI a. C. los fenicios ...
4       Las raíces evolutivas del sufrimiento humano\n...
                              ...                        
8733    El agente Gabriel Syme es reclutado por Scotla...
8734    En esta magnífica obra se cuenta la historia d...
8739    "El bolígrafo de gel verde es el retrato detal...
8743    Cuando una vieja escritora acostumbrada a ment...
8745    ¿Qué futuro tiene un país como España donde la...
Name: sinopsis, Length: 5247, dtype: str

In [53]:
# Normalizamos esos espacios y saltos de línea en la sinopsis para que no afecten a la distancia de Levenshtein
catalogo["sinopsis"] = catalogo["sinopsis"].str.replace("\n", " ")
catalogo["sinopsis"] = catalogo["sinopsis"].str.replace("  ", " ")
catalogo["sinopsis"] = catalogo["sinopsis"].str.replace(" , ", ", ")

## **Duplicados**

In [54]:
# Calculamos cuantos duplicados hay en el catalogo (sin contar el original)
print(f"Duplicados antes de eliminar: {catalogo.duplicated(subset=['sinopsis']).sum()}")

# 1. Detectar los duplicados
duplicados = catalogo[catalogo.duplicated(subset=['sinopsis'], keep=False)]

# 2. Mostrar los casos duplicados
print("Casos duplicados:")
print(duplicados[['titulo', 'sinopsis']])

# 3. Eliminar los duplicados y guardar el resultado
catalogo = catalogo.drop_duplicates(subset=['sinopsis'])

Duplicados antes de eliminar: 98
Casos duplicados:
                                                 titulo  \
192                                 Historia del tiempo   
605   Introducción a la ciencia II. Ciencias biológicas   
615   Historia del tiempo: Del Big Bang a los agujer...   
736   Comentarios psicológicos sobre las enseñanzas ...   
837                                      Viaje al oeste   
...                                                 ...   
8685                                 Alemán – Unidad 43   
8686                                 Alemán – Unidad 42   
8687                                 Alemán – Unidad 41   
8688                                 Alemán – Unidad 16   
8689                                 Alemán – Unidad 17   

                                               sinopsis  
192   Stephen W. Hawking, físico británico (Oxford, ...  
605   Nueva Guía Asimov de la ciencia es un libro pu...  
615   Stephen W. Hawking, físico británico (Oxford, ...  
736   El

In [55]:
# Las nuevas dimensiones de nuestro df son:
catalogo.shape

(8648, 12)

Pasamos de 8746 registros a 8648. Se eliminaron los 98 repetidos.

### **Similares**
Buscamos sinopsis similares (aun que no iguales) con Levenshtein. Elimnamos aquellos registros del mismo autor con mismas sinopsis.

In [ ]:
from itertools import combinations

def encontrar_sinopsis_similares(df, umbral=10):
    """Busca pares de libros del mismo autor cuyas sinopsis difieren en menos de `umbral` caracteres
    (distancia de Levenshtein). Devuelve un DataFrame con un par de libros por fila."""

    # Paso 1: diccionario donde guardamos los pares similares encontrados.
    # Clave: (indice_libro_1, indice_libro_2) -> Valor: distancia entre sus sinopsis.
    # Usar un diccionario evita repetir un par si los dos libros comparten más de un autor.
    pares = {}

    # Paso 2: "explotamos" la columna autores. Si un libro tiene dos autores, pasa a ocupar dos filas
    # (una por autor). Después agrupamos por autor, así cada autor se procesa una sola vez
    # y tenemos juntos todos sus libros.
    for autor, grupo in df.explode("autores").groupby("autores"):

        # Paso 3: si el autor tiene un solo libro no hay con qué comparar, pasamos al siguiente.
        if len(grupo) < 2:
            continue

        # Paso 4: generamos todas las combinaciones posibles de 2 libros del mismo autor.
        # `.items()` devuelve (indice, sinopsis) de cada libro, y `combinations` arma los pares
        # sin repetir ni comparar un libro consigo mismo.
        for (i, sinopsis_i), (j, sinopsis_j) in combinations(grupo["sinopsis"].items(), 2):

            # Paso 5: calculamos la distancia de Levenshtein, es decir, cuántos caracteres hay que
            # insertar, borrar o reemplazar para convertir una sinopsis en la otra.
            # `score_cutoff` corta el cálculo apenas la distancia supera el umbral, lo que acelera
            # mucho la comparación de sinopsis muy distintas.
            distancia = Levenshtein.distance(sinopsis_i, sinopsis_j, score_cutoff=umbral - 1)

            # Paso 6: si la distancia es menor al umbral, las sinopsis son casi iguales y guardamos el par.
            # Ordenamos los índices con min/max para que (3, 7) y (7, 3) cuenten como el mismo par.
            if distancia < umbral:
                pares[(min(i, j), max(i, j))] = distancia

    # Paso 7: por cada par encontrado, armamos una fila con los datos de ambos libros
    # buscándolos en el DataFrame original por su índice.
    filas = []
    for (i, j), distancia in pares.items():
        filas.append({
            "indice_1": i,
            "indice_2": j,
            "autores": df.loc[i, "autores"],
            "titulo_1": df.loc[i, "titulo"],
            "titulo_2": df.loc[j, "titulo"],
            "series_1": df.loc[i, "series"],  # permite ver si ambos libros son de una misma saga
            "series_2": df.loc[j, "series"],
            "distancia": distancia,
        })

    # Paso 8: convertimos las filas en un DataFrame y lo ordenamos de menor a mayor distancia,
    # así los pares más parecidos quedan arriba.
    return pd.DataFrame(filas).sort_values("distancia").reset_index(drop=True)

# Ejecutamos la función sobre el catálogo y guardamos el resultado en una variable
sinopsis_similares = encontrar_sinopsis_similares(catalogo, umbral=10)

print(f"Pares de libros con sinopsis similares: {len(sinopsis_similares)}")
sinopsis_similares.head(10)


Pares de libros con sinopsis similares: 38


,indice_1,indice_2,autores,titulo_1,titulo_2,series_1,series_2,distancia
0,1250,8263,[Arturo Pérez-Reverte],"Patente de corso, 2015","Patente de corso, 2013",NaN,NaN,1
1,6023,8216,[Benito Jerónimo Feijoo],"Cartas eruditas, y curiosas II","Cartas eruditas, y curiosas V",Cartas eruditas y curiosas,Cartas eruditas y curiosas,1
2,206,6001,[Isaac Asimov],El secreto del universo,El secreto del universo y otros ensayos cientí...,NaN,NaN,1
3,7443,7446,[Benito Jerónimo Feijoo],Teatro crítico universal I,Teatro crítico universal II,Teatro crítico universal,Teatro crítico universal,1
4,3507,7219,[Javier Marías],"La zona fantasma, 2016","La zona fantasma, 2017",NaN,NaN,1
5,2435,2438,"[Javier Olivares, Pablo Olivares]",Tiempo de gloria. El Ministerio del Tiempo 02,El tiempo es el que es. El ministerio del tiem...,El Ministerio del Tiempo,El Ministerio del Tiempo,1
6,1249,8264,[Javier Marías],"La zona fantasma, 2015","La zona fantasma, 2013",NaN,NaN,1
7,5165,5167,[Thomas Mann],Las historias de Jaacob,José el proveedor,José y sus hermanos,José y sus hermanos,1
8,5167,5168,[Thomas Mann],José el proveedor,El joven José,José y sus hermanos,José y sus hermanos,1
9,2297,5042,[Julio Caro Baroja],Vidas mágicas e Inquisición I,Vidas mágicas e Inquisición II,NaN,NaN,2
